# Part A: Concept Application

## Imports Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance

## Function to Create Synthetic Data

In [20]:
def create_loan_data(n=2000):
    np.random.seed(42)

    annual_income = np.random.randint(20000, 150000, n)
    credit_score = np.random.randint(300, 850, n)
    loan_amount = np.random.randint(5000, 50000, n)
    employment_years = np.random.randint(0, 30, n)
    debt_to_income = np.round(np.random.uniform(0.1, 0.8, n), 2)
    num_credit_cards = np.random.randint(1, 10, n)

    approved = (
        (credit_score > 700) &
        (debt_to_income < 0.3) &
        (annual_income > 40000) &
        (employment_years > 2)
    ).astype(int)

    noise = np.random.choice([0, 1], size=n, p=[0.9, 0.1])
    approved = np.where(noise == 1, 1 - approved, approved)

    return pd.DataFrame({
        'annual_income': annual_income,
        'credit_score': credit_score,
        'loan_amount': loan_amount,
        'employment_years': employment_years,
        'debt_to_income': debt_to_income,
        'num_credit_cards': num_credit_cards,
        'approved': approved
    })


In [21]:
df = create_loan_data()
df.head()

,annual_income,credit_score,loan_amount,employment_years,debt_to_income,num_credit_cards,approved
0,141958,499,33884,21,0.62,3,0
1,35795,836,29885,4,0.13,1,0
2,20860,765,7708,22,0.76,8,0
3,123694,388,23375,4,0.60,5,0
4,148106,567,20088,9,0.10,4,0


## Train-Test Split

In [22]:
X = df.drop('approved', axis=1)
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Evaluate Model

In [23]:
def evaluate_model(model, X_test, y_test):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, prob)

    return accuracy, f1, auc

## Decision Tree Training

In [24]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

dt_accuracy, dt_f1, dt_auc = evaluate_model(dt, X_test, y_test)

print("Decision Tree Accuracy:", dt_accuracy)
print("Decision Tree F1 Score:", dt_f1)
print("Decision Tree ROC-AUC:", dt_auc)

Decision Tree Accuracy: 0.895
Decision Tree F1 Score: 0.4878048780487805
Decision Tree ROC-AUC: 0.6699145823743031


## Decision Rules

In [25]:
rules = export_text(dt, feature_names=list(X.columns))
print(rules)

|--- credit_score <= 707.50
|   |--- annual_income <= 20088.50
|   |   |--- class: 1
|   |--- annual_income >  20088.50
|   |   |--- loan_amount <= 5097.00
|   |   |   |--- class: 1
|   |   |--- loan_amount >  5097.00
|   |   |   |--- loan_amount <= 44429.00
|   |   |   |   |--- class: 0
|   |   |   |--- loan_amount >  44429.00
|   |   |   |   |--- class: 0
|--- credit_score >  707.50
|   |--- debt_to_income <= 0.30
|   |   |--- annual_income <= 39256.00
|   |   |   |--- credit_score <= 717.50
|   |   |   |   |--- class: 1
|   |   |   |--- credit_score >  717.50
|   |   |   |   |--- class: 0
|   |   |--- annual_income >  39256.00
|   |   |   |--- employment_years <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- employment_years >  2.50
|   |   |   |   |--- class: 1
|   |--- debt_to_income >  0.30
|   |   |--- loan_amount <= 49949.00
|   |   |   |--- credit_score <= 710.00
|   |   |   |   |--- class: 1
|   |   |   |--- credit_score >  710.00
|   |   |   |   |--- class: 0
|   |   |

## Random Forest

In [26]:
def train_random_forest(X_train, y_train):
    rf = RandomForestClassifier(random_state=42)

    param_dist = {
        'n_estimators': [50, 100, 150],
        'max_depth': [3, 5, 7, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    search = RandomizedSearchCV(
        rf,
        param_distributions=param_dist,
        n_iter=10,
        cv=5,
        scoring='f1',
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_

best_rf, best_params = train_random_forest(X_train, y_train)

for key, value in best_params.items():
    print(f"{key}: {value}")

n_estimators: 100
min_samples_split: 2
min_samples_leaf: 1
max_depth: None


## Random Forest Evaluation

In [28]:
rf_accuracy, rf_f1, rf_auc = evaluate_model(best_rf, X_test, y_test)

print("Random Forest Accuracy:", rf_accuracy * 100, "%")
print("Random Forest F1 Score:", rf_f1)
print("Random Forest ROC-AUC:", rf_auc)

Random Forest Accuracy: 90.0 %
Random Forest F1 Score: 0.5
Random Forest ROC-AUC: 0.667817502941026


## Feature Importance Function

In [29]:
def show_feature_importance(model, X, X_test, y_test):
    default_imp = pd.Series(
        model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)

    perm_imp = permutation_importance(
        model, X_test, y_test, n_repeats=10, random_state=42
    )

    perm_series = pd.Series(
        perm_imp.importances_mean,
        index=X.columns
    ).sort_values(ascending=False)

    return default_imp, perm_series

default_imp, perm_imp = show_feature_importance(best_rf, X, X_test, y_test)

print("Default Importance:")
print(default_imp)

print("\nPermutation Importance:")
print(perm_imp)

Default Importance:
credit_score        0.261762
debt_to_income      0.217310
annual_income       0.179139
loan_amount         0.153132
employment_years    0.118862
num_credit_cards    0.069795
dtype: float64

Permutation Importance:
debt_to_income      0.07775
credit_score        0.07750
annual_income       0.01325
employment_years    0.00475
loan_amount        -0.00050
num_credit_cards   -0.00100
dtype: float64


## Comparison Table

In [16]:
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [dt_accuracy, rf_accuracy],
    'F1 Score': [dt_f1, rf_f1],
    'ROC-AUC': [dt_auc, rf_auc],
    'Interpretability': ['High', 'Medium-Low']
})

comparison

,Model,Accuracy,F1 Score,ROC-AUC,Interpretability
0,Decision Tree,0.9000,0.672131,0.774605,High
1,Random Forest,0.9025,0.677686,0.745477,Medium-Low


### Recommendation:
Decision Tree offers excellent interpretability because each loan decision
can be directly explained using simple rules, which helps regulatory compliance.

Random Forest generally achieves better predictive performance because it reduces
overfitting through ensemble learning.

Best deployment strategy:
Use Random Forest for final approval prediction and Decision Tree for explanation
to regulators and internal audit teams.

# Part B:- Stretch Problem